# Piper Voice Batch Generator

This notebook loads location instructions, synthesizes speech files with Piper, and updates each entry with the generated audio file path.


In [2]:
!pip install piper-tts

  Using cached piper_tts-1.3.0-cp39-abi3-win_amd64.whl.metadata (4.5 kB)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
  Using cached pyreadline3-3.5.4-py3-none-any.whl.metadata (4.7 kB)
Using cached piper_tts-1.3.0-cp39-abi3-win_amd64.whl (13.8 MB)
   ---------------------------------------- 0.0/13.5 MB ? eta -:--:--
   ---------------------------- ----------- 9.4/13.5 MB 49.1 MB/s eta 0:00:01
   ---------------------------------------- 13.5/13.5 MB 56.4 MB/s  0:00:00
Using cached coloredlogs-15.0.1-py2.py3-none-any.whl (46 kB)
Using cached humanfriendly-10.0-py2.py3-none-any.whl (86 kB)
Using cached pyreadline3-3.5.4-py3-none-any.whl (83 kB)

   ------------------------ --------------- 3/5 [onnxruntime]
   ------------------------ --------------- 3/5 [onnxruntime]
   ------------------------ --------------- 3/5 [onnxruntime]
   ------------------------ --------------- 3/5 [onnxruntime]


In [ ]:
from __future__ import annotations

import json
import os
import re
import wave
from pathlib import Path
from typing import Optional
import piper
from piper import PiperVoice, SynthesisConfig




def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (candidate / "touchscreen-display").exists():
            return candidate
    return start


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

INSTRUCTIONS_PATH = REPO_ROOT / "touchscreen-display" / "public" / "instructions.json"
AUDIO_ROOT = REPO_ROOT / "touchscreen-display" / "public"

OUTPUT_AUDIO_RELATIVE = Path("audio") / "instructions"
OUTPUT_AUDIO_DIR = AUDIO_ROOT / OUTPUT_AUDIO_RELATIVE

VOICE_PATH = REPO_ROOT / "chatbot" / "app" / "voices" / "en_US-amy-medium.onnx"
VOICE_CONFIG_PATH = VOICE_PATH.with_suffix(VOICE_PATH.suffix + ".json")

USE_CUDA = True
SPEAKER_ID: Optional[int] = None
LENGTH_SCALE: Optional[float] = None
NOISE_SCALE: Optional[float] = None
NOISE_W_SCALE: Optional[float] = None
VOLUME: float = 1.0

OVERWRITE_AUDIO = False
AUDIO_FILENAME_TEMPLATE = "{index:03d}_{slug}.wav"


In [4]:
def slugify(value: str, fallback: str, max_length: int = 48) -> str:
    value = value or ""
    slug = re.sub(r"[^a-z0-9]+", "-", value.lower())
    slug = slug.strip("-") or fallback
    return slug[:max_length].rstrip("-") or fallback


def ensure_dependencies() -> None:
    missing: list[str] = []
    if not INSTRUCTIONS_PATH.exists():
        missing.append(f"Missing instructions file: {INSTRUCTIONS_PATH}")
    if not VOICE_PATH.exists():
        missing.append(f"Missing Piper voice model: {VOICE_PATH}")
    if not VOICE_CONFIG_PATH.exists():
        missing.append(f"Missing Piper voice config: {VOICE_CONFIG_PATH}")
    if missing:
        raise FileNotFoundError("\n".join(missing))
    OUTPUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)


ensure_dependencies()
voice = PiperVoice.load(str(VOICE_PATH), str(VOICE_CONFIG_PATH), use_cuda=USE_CUDA)


def build_synthesis_config() -> SynthesisConfig:
    return SynthesisConfig(
        speaker_id=SPEAKER_ID,
        length_scale=float(LENGTH_SCALE) if LENGTH_SCALE is not None else None,
        noise_scale=float(NOISE_SCALE) if NOISE_SCALE is not None else None,
        noise_w_scale=float(NOISE_W_SCALE) if NOISE_W_SCALE is not None else None,
        volume=float(VOLUME),
    )


voice, build_synthesis_config()


(PiperVoice(session=<onnxruntime.capi.onnxruntime_inference_collection.InferenceSession object at 0x000002B2FFEFA490>, config=PiperConfig(num_symbols=256, num_speakers=1, sample_rate=22050, espeak_voice='en-us', phoneme_id_map={'_': [0], '^': [1], '$': [2], ' ': [3], '!': [4], "'": [5], '(': [6], ')': [7], ',': [8], '-': [9], '.': [10], ':': [11], ';': [12], '?': [13], 'a': [14], 'b': [15], 'c': [16], 'd': [17], 'e': [18], 'f': [19], 'h': [20], 'i': [21], 'j': [22], 'k': [23], 'l': [24], 'm': [25], 'n': [26], 'o': [27], 'p': [28], 'q': [29], 'r': [30], 's': [31], 't': [32], 'u': [33], 'v': [34], 'w': [35], 'x': [36], 'y': [37], 'z': [38], 'æ': [39], 'ç': [40], 'ð': [41], 'ø': [42], 'ħ': [43], 'ŋ': [44], 'œ': [45], 'ǀ': [46], 'ǁ': [47], 'ǂ': [48], 'ǃ': [49], 'ɐ': [50], 'ɑ': [51], 'ɒ': [52], 'ɓ': [53], 'ɔ': [54], 'ɕ': [55], 'ɖ': [56], 'ɗ': [57], 'ɘ': [58], 'ə': [59], 'ɚ': [60], 'ɛ': [61], 'ɜ': [62], 'ɞ': [63], 'ɟ': [64], 'ɠ': [65], 'ɡ': [66], 'ɢ': [67], 'ɣ': [68], 'ɤ': [69], 'ɥ': [70], '

In [5]:
with INSTRUCTIONS_PATH.open("r", encoding="utf-8") as fh:
    instructions: list[dict[str, object]] = json.load(fh)

updated_instructions: list[dict[str, object]] = []
synthesis_config = build_synthesis_config()

for idx, entry in enumerate(instructions, start=1):
    current = dict(entry)
    directions = str(current.get("directions", "")).strip()

    if not directions:
        current["file_path"] = ""
        updated_instructions.append(current)
        print(f"[skip] No directions for index {idx}; file_path left empty")
        continue

    slug_source = str(current.get("location") or current.get("code") or f"entry-{idx}")
    slug = slugify(slug_source, fallback=f"entry-{idx}")
    filename = AUDIO_FILENAME_TEMPLATE.format(index=idx, slug=slug)
    audio_path = OUTPUT_AUDIO_DIR / filename

    if audio_path.exists() and not OVERWRITE_AUDIO:
        print(f"[skip] {filename} already exists")
    else:
        print(f"[piper] Synthesizing {filename}")
        with wave.open(str(audio_path), "wb") as wav_file:
            voice.synthesize_wav(
                directions,
                wav_file,
                syn_config=synthesis_config,
            )

    relative_path = (OUTPUT_AUDIO_RELATIVE / filename).as_posix()
    current["file_path"] = relative_path
    updated_instructions.append(current)

with INSTRUCTIONS_PATH.open("w", encoding="utf-8") as fh:
    json.dump(updated_instructions, fh, indent=2, ensure_ascii=False)
    fh.write("\n")

print(f"Updated {len(updated_instructions)} instructions.")


[piper] Synthesizing 001_diagnostic-imaging-2.wav
[piper] Synthesizing 002_urgent-care-centre.wav
[piper] Synthesizing 003_ward-8.wav
[piper] Synthesizing 004_ward-9.wav
[piper] Synthesizing 005_ward-10.wav
[piper] Synthesizing 006_ward-11.wav
[piper] Synthesizing 007_diagnostic-imaging-3.wav
[piper] Synthesizing 008_major-operating-theatres-1-2.wav
[piper] Synthesizing 009_intensive-care-unit-1.wav
[piper] Synthesizing 010_major-operating-theatres-3-4.wav
[piper] Synthesizing 011_ward-12.wav
[piper] Synthesizing 012_ward-13.wav
[piper] Synthesizing 013_clinical-measurement-centre.wav
[piper] Synthesizing 014_pharmacy.wav
[piper] Synthesizing 015_clinic-j.wav
[piper] Synthesizing 016_clinic-k.wav
[piper] Synthesizing 017_ward-7.wav
[piper] Synthesizing 018_care-and-counselling.wav
[piper] Synthesizing 019_ear-nose-and-throat-centre.wav
[piper] Synthesizing 020_eye-surgery-centre.wav
[piper] Synthesizing 021_surgery-centre.wav
[piper] Synthesizing 022_ambulatory-surgery-centre.wav
[pipe